In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

events = pd.read_csv("events.csv")

events.head()

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [ ]:
events.shape

(2756101, 5)

In [ ]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2756101 entries, 0 to 2756100
Data columns (total 5 columns):
 #   Column         Dtype  
---  ------         -----  
 0   timestamp      int64  
 1   visitorid      int64  
 2   event          object 
 3   itemid         int64  
 4   transactionid  float64
dtypes: float64(1), int64(3), object(1)
memory usage: 105.1+ MB


In [ ]:
events["event"].value_counts()

,count
event,
view,2664312
addtocart,69332
transaction,22457


In [ ]:
events = pd.read_csv("events.csv")

events["timestamp"] = pd.to_datetime(
    events["timestamp"],
    unit="ms"
)

events.head()

,timestamp,visitorid,event,itemid,transactionid
0,2015-06-02 05:02:12.117,257597,view,355908,NaN
1,2015-06-02 05:50:14.164,992329,view,248676,NaN
2,2015-06-02 05:13:19.827,111016,view,318965,NaN
3,2015-06-02 05:12:35.914,483717,view,253185,NaN
4,2015-06-02 05:02:17.106,951259,view,367447,NaN


In [ ]:
print("Start:", events["timestamp"].min())
print("End:", events["timestamp"].max())
print("Duration:", events["timestamp"].max() - events["timestamp"].min())

Start: 2015-05-03 03:00:04.384000
End: 2015-09-18 02:59:47.788000
Duration: 137 days 23:59:43.404000


In [ ]:
print("Total events:", len(events))
print("Unique visitors:", events["visitorid"].nunique())
print("Unique products:", events["itemid"].nunique())

Total events: 2756101
Unique visitors: 1407580
Unique products: 235061


In [ ]:
events["event"].value_counts()

,count
event,
view,2664312
addtocart,69332
transaction,22457


In [ ]:
events.isna().sum()

,0
timestamp,0
visitorid,0
event,0
itemid,0
transactionid,2733644


In [ ]:
visitor_events = (
    events.groupby("visitorid")["event"]
    .agg(list)
)

visitor_events.head()

,event
visitorid,
0,"[view, view, view]"
1,[view]
2,"[view, view, view, view, view, view, view, view]"
3,[view]
4,[view]


In [ ]:
def get_event_sequence(events_list):
    return " → ".join(events_list)

sequences = visitor_events.apply(get_event_sequence)

sequences.value_counts().head(20)

,count
event,
view,998953
view → view,199135
view → view → view,73296
view → view → view → view,34259
view → view → view → view → view,19683
view → view → view → view → view → view,11646
view → view → view → view → view → view → view,7600
view → view → view → view → view → view → view → view,5346
view → view → view → view → view → view → view → view → view,3752


In [ ]:
import pandas as pd

# Load data
events = pd.read_csv("events.csv")

# Convert Unix timestamp (milliseconds) to datetime
events["timestamp"] = pd.to_datetime(events["timestamp"], unit="ms")

# Sort chronologically for each visitor
events = events.sort_values(["visitorid", "timestamp"]).reset_index(drop=True)

# Calculate time since previous event for each visitor
events["time_diff"] = (
    events.groupby("visitorid")["timestamp"].diff()
)

# Start a new session after 30 minutes of inactivity
events["new_session"] = (
    events["time_diff"].isna()
    | (events["time_diff"] > pd.Timedelta(minutes=30))
)

# Assign session number within each visitor
events["session_id"] = (
    events.groupby("visitorid")["new_session"].cumsum()
)

# Basic session statistics
session_count = events[["visitorid", "session_id"]].drop_duplicates().shape[0]

print(f"Total events: {len(events):,}")
print(f"Unique visitors: {events['visitorid'].nunique():,}")
print(f"Unique sessions: {session_count:,}")

# Inspect a few session sequences
session_sequences = (
    events.groupby(["visitorid", "session_id"])["event"]
    .agg(list)
)

print("\nSample session sequences:")
print(session_sequences.head(20))

Total events: 2,756,101
Unique visitors: 1,407,580
Unique sessions: 1,761,675

Sample session sequences:
visitorid  session_id
0          1                                           [view, view, view]
1          1                                                       [view]
2          1             [view, view, view, view, view, view, view, view]
3          1                                                       [view]
4          1                                                       [view]
5          1                                                       [view]
6          1                                                  [addtocart]
           2                                     [view, view, view, view]
           3                                                       [view]
7          1                                                 [view, view]
           2                                                       [view]
8          1                                               

In [ ]:
# Build one row per session efficiently

session_groups = events.groupby(["visitorid", "session_id"])

sessions = session_groups.agg(
    session_start=("timestamp", "min"),
    session_end=("timestamp", "max"),
    event_count=("event", "size"),
    unique_products=("itemid", "nunique"),
).reset_index()

# Count each event type using a vectorized pivot
event_counts = (
    events.assign(count=1)
    .pivot_table(
        index=["visitorid", "session_id"],
        columns="event",
        values="count",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

# Remove the column-name label created by pivot_table
event_counts.columns.name = None

# Rename event counts
event_counts = event_counts.rename(columns={
    "view": "views",
    "addtocart": "cart_additions",
    "transaction": "transactions"
})

# Merge event counts into session data
sessions = sessions.merge(
    event_counts,
    on=["visitorid", "session_id"],
    how="left"
)

# Make sure missing event types are represented as zero
for col in ["views", "cart_additions", "transactions"]:
    if col not in sessions:
        sessions[col] = 0

# Derived session metrics
sessions["duration_minutes"] = (
    sessions["session_end"] - sessions["session_start"]
).dt.total_seconds() / 60

sessions["has_cart"] = sessions["cart_additions"] > 0
sessions["has_transaction"] = sessions["transactions"] > 0

# Preliminary abandonment definition
sessions["abandoned"] = (
    sessions["has_cart"] & ~sessions["has_transaction"]
)

# Summary
print(f"Total sessions: {len(sessions):,}")
print(f"Sessions with cart activity: {sessions['has_cart'].sum():,}")
print(f"Sessions with transactions: {sessions['has_transaction'].sum():,}")
print(f"Preliminary abandoned sessions: {sessions['abandoned'].sum():,}")

print("\nAbandonment rate among cart sessions:",
      f"{sessions.loc[sessions['has_cart'], 'abandoned'].mean() * 100:.2f}%")

print("\nSession sample:")
print(sessions.head(10).to_string(index=False))

Total sessions: 1,761,675
Sessions with cart activity: 43,924
Sessions with transactions: 14,297
Preliminary abandoned sessions: 31,992

Abandonment rate among cart sessions: 72.83%

Session sample:
 visitorid  session_id           session_start             session_end  event_count  unique_products  cart_additions  transactions  views  duration_minutes  has_cart  has_transaction  abandoned
         0           1 2015-09-11 20:49:49.439 2015-09-11 20:55:17.175            3                3               0             0      3          5.462267     False            False      False
         1           1 2015-08-13 17:46:06.444 2015-08-13 17:46:06.444            1                1               0             0      1          0.000000     False            False      False
         2           1 2015-08-07 17:51:44.567 2015-08-07 18:20:57.845            8                4               0             0      8         29.221300     False            False      False
         3           1 20

In [ ]:
# Identify abandoned checkout sessions
abandoned = sessions[sessions["abandoned"]].copy()

# Keep the information available at abandonment time
abandoned = abandoned[
    [
        "visitorid",
        "session_id",
        "session_end",
        "cart_additions",
        "views",
        "unique_products",
        "event_count",
        "duration_minutes"
    ]
].rename(columns={
    "session_end": "abandoned_at"
})

# For each abandoned session, find the visitor's first transaction
# occurring after the abandonment timestamp.
transactions = (
    events[events["event"] == "transaction"]
    [["visitorid", "timestamp", "itemid", "transactionid"]]
    .rename(columns={"timestamp": "transaction_time"})
)

recovery = abandoned.merge(
    transactions,
    on="visitorid",
    how="left"
)

# Keep only transactions occurring after abandonment
recovery["days_after_abandonment"] = (
    recovery["transaction_time"] - recovery["abandoned_at"]
).dt.total_seconds() / (24 * 60 * 60)

recovery = recovery[
    (recovery["days_after_abandonment"] > 0)
    & (recovery["days_after_abandonment"] <= 7)
]

# One recovery outcome per abandoned session
recovered_sessions = (
    recovery.groupby(["visitorid", "session_id"])
    .agg(
        recovery_time=("transaction_time", "min"),
        recovery_days=("days_after_abandonment", "min")
    )
    .reset_index()
)

# Add the recovery label
abandoned = abandoned.merge(
    recovered_sessions,
    on=["visitorid", "session_id"],
    how="left"
)

abandoned["recovered_within_7d"] = (
    abandoned["recovery_time"].notna()
).astype(int)

# Summary
total_abandoned = len(abandoned)
total_recovered = abandoned["recovered_within_7d"].sum()

print(f"Abandoned sessions: {total_abandoned:,}")
print(f"Recovered within 7 days: {total_recovered:,}")
print(
    f"7-day recovery rate: "
    f"{total_recovered / total_abandoned * 100:.2f}%"
)

print("\nRecovery distribution:")
print(
    abandoned["recovered_within_7d"]
    .value_counts()
    .rename({0: "Not recovered", 1: "Recovered"})
)

print("\nRecovered session sample:")
print(
    abandoned[abandoned["recovered_within_7d"] == 1]
    .head(10)
    .to_string(index=False)
)

Abandoned sessions: 31,992
Recovered within 7 days: 1,996
7-day recovery rate: 6.24%

Recovery distribution:
recovered_within_7d
Not recovered    29996
Recovered         1996
Name: count, dtype: int64

Recovered session sample:
 visitorid  session_id            abandoned_at  cart_additions  views  unique_products  event_count  duration_minutes           recovery_time  recovery_days  recovered_within_7d
       419           2 2015-07-29 04:03:49.136               1      2                3            3         15.925083 2015-07-29 05:03:12.695       0.041245                    1
      1032           2 2015-06-26 16:20:58.601               1      1                1            2          5.714817 2015-06-26 19:28:50.661       0.130464                    1
      2019           2 2015-06-03 06:30:54.091               1      6                5            7         22.848817 2015-06-03 21:55:09.996       0.641851                    1
      2019           3 2015-06-03 07:47:07.474              

In [ ]:
# ---------------------------------------------------------
# Build a clean checkout recovery dataset
# ---------------------------------------------------------

# Dataset end date
data_end = events["timestamp"].max()

# Start from abandoned sessions
checkout = sessions[sessions["abandoned"]].copy()

checkout = checkout[
    [
        "visitorid",
        "session_id",
        "session_start",
        "session_end",
        "cart_additions",
        "views",
        "unique_products",
        "event_count",
        "duration_minutes"
    ]
].rename(columns={
    "session_end": "abandoned_at"
})

# Only keep sessions where we can observe the complete
# 7-day recovery window
checkout["observation_end"] = (
    checkout["abandoned_at"] + pd.Timedelta(days=7)
)

checkout = checkout[
    checkout["observation_end"] <= data_end
].copy()

# ---------------------------------------------------------
# Find transactions after abandonment
# ---------------------------------------------------------

transactions = (
    events[events["event"] == "transaction"]
    [
        ["visitorid", "timestamp", "itemid", "transactionid"]
    ]
    .rename(columns={
        "timestamp": "transaction_time"
    })
)

recovery_candidates = checkout.merge(
    transactions,
    on="visitorid",
    how="left"
)

# Time between abandonment and transaction
recovery_candidates["days_after_abandonment"] = (
    recovery_candidates["transaction_time"]
    - recovery_candidates["abandoned_at"]
).dt.total_seconds() / (24 * 60 * 60)

# Only transactions within the 7-day window
recovery_candidates = recovery_candidates[
    (recovery_candidates["days_after_abandonment"] > 0)
    & (recovery_candidates["days_after_abandonment"] <= 7)
].copy()

# ---------------------------------------------------------
# Assign each transaction to the closest preceding
# abandoned session for that visitor.
# This prevents one transaction from recovering
# multiple abandoned sessions.
# ---------------------------------------------------------

recovery_candidates = recovery_candidates.sort_values(
    ["visitorid", "transaction_time", "abandoned_at"]
)

recovery_candidates["transaction_rank"] = (
    recovery_candidates
    .groupby(["visitorid", "transaction_time"])
    .cumcount()
)

# Closest abandoned session before each transaction
recovered = (
    recovery_candidates
    .sort_values("days_after_abandonment")
    .drop_duplicates(
        subset=["visitorid", "transaction_time"],
        keep="first"
    )
)

# Each abandoned session gets at most one recovery
recovered = (
    recovered
    .sort_values("days_after_abandonment")
    .drop_duplicates(
        subset=["visitorid", "session_id"],
        keep="first"
    )
)

# ---------------------------------------------------------
# Add recovery outcome
# ---------------------------------------------------------

checkout = checkout.merge(
    recovered[
        [
            "visitorid",
            "session_id",
            "transaction_time",
            "days_after_abandonment"
        ]
    ],
    on=["visitorid", "session_id"],
    how="left"
)

checkout["recovered_within_7d"] = (
    checkout["transaction_time"].notna()
).astype(int)

# ---------------------------------------------------------
# Final summary
# ---------------------------------------------------------

print("CLEAN CHECKOUT RECOVERY DATASET")
print("-" * 40)

print(f"Abandoned sessions with full 7-day window: {len(checkout):,}")
print(
    f"Recovered within 7 days: "
    f"{checkout['recovered_within_7d'].sum():,}"
)

print(
    f"7-day recovery rate: "
    f"{checkout['recovered_within_7d'].mean() * 100:.2f}%"
)

print("\nTarget distribution:")
print(
    checkout["recovered_within_7d"]
    .value_counts()
    .rename({
        0: "Not recovered",
        1: "Recovered"
    })
)

print("\nRecovery timing:")
print(
    checkout.loc[
        checkout["recovered_within_7d"] == 1,
        "days_after_abandonment"
    ].describe()
)

print("\nFinal columns:")
print(checkout.columns.tolist())

CLEAN CHECKOUT RECOVERY DATASET
----------------------------------------
Abandoned sessions with full 7-day window: 30,618
Recovered within 7 days: 1,654
7-day recovery rate: 5.40%

Target distribution:
recovered_within_7d
Not recovered    28964
Recovered         1654
Name: count, dtype: int64

Recovery timing:
count    1654.000000
mean        0.761144
std         1.360427
min         0.020848
25%         0.042759
50%         0.126879
75%         0.818815
max         6.980208
Name: days_after_abandonment, dtype: float64

Final columns:
['visitorid', 'session_id', 'session_start', 'abandoned_at', 'cart_additions', 'views', 'unique_products', 'event_count', 'duration_minutes', 'observation_end', 'transaction_time', 'days_after_abandonment', 'recovered_within_7d']


In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

# 1. Final Test Set Evaluation
rf_model = joblib.load('selected_recovery_model.pkl')
test_probs = rf_model.predict_proba(X_test)[:, 1]
test_preds = rf_model.predict(X_test)

print("FINAL TEST SET EVALUATION")
print("-" * 30)
print(f"ROC-AUC: {roc_auc_score(y_test, test_probs):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, test_probs):.4f}")
print(classification_report(y_test, test_preds))

# 2. Expected Recovery Analysis (Business Value)
# Assumptions for simulation:
# - Average Order Value (AOV): $100
# - Intervention (e.g., email discount) increases recovery chance by 20% (relative)

aov = 100
lift = 0.20

test_results = pd.DataFrame({
    'actual': y_test,
    'prob': test_probs
})

# Revenue at risk = sessions where model predicts high probability but hasn't recovered yet
# We group by deciles to see where intervention is most efficient
test_results['decile'] = pd.qcut(test_results['prob'], 10, labels=False)

decile_analysis = test_results.groupby('decile').agg(
    sessions=('actual', 'count'),
    actual_recoveries=('actual', 'sum'),
    avg_prob=('prob', 'mean')
).sort_index(ascending=False)

decile_analysis['expected_revenue'] = decile_analysis['actual_recoveries'] * aov
decile_analysis['potential_lift_revenue'] = decile_analysis['expected_revenue'] * lift

print("\nBUSINESS VALUE ANALYSIS (By Probability Decile)")
print("-" * 50)
print(decile_analysis)

# 3. Final Summary
total_potential_lift = decile_analysis['potential_lift_revenue'].sum()
print(f"\nTotal estimated 7-day revenue at risk in test set: ${decile_analysis['expected_revenue'].sum():,.2f}")
print(f"Estimated revenue recovery lift from intervention: ${total_potential_lift:,.2f}")

FINAL TEST SET EVALUATION
------------------------------
ROC-AUC: 0.5954
PR-AUC:  0.0871
              precision    recall  f1-score   support

           0       0.96      0.87      0.91      4367
           1       0.08      0.22      0.12       226

    accuracy                           0.84      4593
   macro avg       0.52      0.55      0.52      4593
weighted avg       0.91      0.84      0.87      4593


BUSINESS VALUE ANALYSIS (By Probability Decile)
--------------------------------------------------
        sessions  actual_recoveries  avg_prob  expected_revenue  \
decile                                                            
9            460                 40  0.591750              4000   
8            459                 33  0.490557              3300   
7            459                 28  0.442574              2800   
6            456                 18  0.406935              1800   
5            462                 20  0.378133              2000   
4            46

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import roc_auc_score, average_precision_score

# 1. Load existing model and evaluate on test set
rf_model = joblib.load('selected_recovery_model.pkl')
test_probs = rf_model.predict_proba(X_test)[:, 1]

# Metrics
test_count = len(y_test)
pos_count = y_test.sum()
overall_rate = pos_count / test_count
roc_auc = roc_auc_score(y_test, test_probs)
pr_auc = average_precision_score(y_test, test_probs)

print(f"--- Test Set Evaluation ---")
print(f"Test-set sample count: {test_count}")
print(f"Positive/recovered count: {pos_count}")
print(f"Overall recovery rate: {overall_rate:.2%}")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}\n")

# 2. Probability-decile analysis
analysis_df = pd.DataFrame({
    'actual': y_test,
    'prob': test_probs
})

# Rank by probability descending
analysis_df = analysis_df.sort_values('prob', ascending=False).reset_index(drop=True)
analysis_df['decile'] = pd.qcut(analysis_df.index, 10, labels=False) + 1

decile_stats = analysis_df.groupby('decile').agg(
    sessions=('actual', 'count'),
    avg_prob=('prob', 'mean'),
    actual_recoveries=('actual', 'sum')
).reset_index()

decile_stats['actual_recovery_rate'] = decile_stats['actual_recoveries'] / decile_stats['sessions']

# Cumulative stats
decile_stats['cum_sessions'] = decile_stats['sessions'].cumsum()
decile_stats['cum_recoveries'] = decile_stats['actual_recoveries'].cumsum()
decile_stats['cum_recovery_rate'] = decile_stats['cum_recoveries'] / decile_stats['cum_sessions']

# 3. Lift Calculation
baseline_rate = 0.054  # Based on the provided overall dataset rate
decile_stats['decile_lift'] = decile_stats['actual_recovery_rate'] / baseline_rate

display(decile_stats)

# Specific top-tier lift
def get_top_k_lift(k_percent):
    count = int(test_count * (k_percent / 100))
    top_recoveries = analysis_df.iloc[:count]['actual'].sum()
    rate = top_recoveries / count
    return rate / baseline_rate

print("\n--- Concentration Lift Analysis ---")
for k in [10, 20, 30, 50]:
    print(f"Top {k}% Lift: {get_top_k_lift(k):.2f}x")

--- Test Set Evaluation ---
Test-set sample count: 4593
Positive/recovered count: 226
Overall recovery rate: 4.92%
ROC-AUC: 0.5954
PR-AUC:  0.0871



,decile,sessions,avg_prob,actual_recoveries,actual_recovery_rate,cum_sessions,cum_recoveries,cum_recovery_rate,decile_lift
0,1,460,0.591750,40,0.086957,460,40,0.086957,1.610306
1,2,459,0.490557,33,0.071895,919,73,0.079434,1.331397
2,3,459,0.442574,28,0.061002,1378,101,0.073295,1.129670
3,4,459,0.406839,18,0.039216,1837,119,0.064780,0.726216
4,5,460,0.378005,20,0.043478,2297,139,0.060514,0.805153
5,6,459,0.345291,17,0.037037,2756,156,0.056604,0.685871
6,7,459,0.309852,29,0.063181,3215,185,0.057543,1.170015
7,8,459,0.267678,12,0.026144,3674,197,0.053620,0.484144
8,9,459,0.210327,18,0.039216,4133,215,0.052020,0.726216
9,10,460,0.121214,11,0.023913,4593,226,0.049205,0.442834



--- Concentration Lift Analysis ---
Top 10% Lift: 1.61x
Top 20% Lift: 1.47x
Top 30% Lift: 1.36x
Top 50% Lift: 1.12x


### 4. Business Interpretation

*   **Prioritization Utility:** The model effectively ranks sessions by natural recovery propensity. The Top 10% (Decile 1) shows a recovery rate significantly higher than the baseline, allowing the intervention engine to distinguish between "high-propensity" and "low-propensity" abandoners.
*   **Recovery Signal:** The lift in the upper deciles confirms that the model captures session behaviors (like duration and view count) that correlate with organic purchase completion. A lift above 1.0x indicates the model is better than random at identifying where organic conversion is likely.
*   **Causal Limitations:** This model measures **propensity to recover organically**, not **sensitivity to intervention**. We do not yet know if a high-propensity user is *saved* by an intervention or if they would have bought anyway (cannibalization). Conversely, low-propensity users might be the most "persuadable," but this requires randomized testing (A/B testing) to establish.

### 5. Final Recommendation

**KEEP** — The model provides a sufficiently useful ranking signal for the recovery decision layer to prioritize high-intent abandoners for specific types of intervention (e.g., low-friction reminders) versus low-intent abandoners who might require stronger incentives.